In [22]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

from ingest import build_index
from rag_helper import RAGHW
from utilities import calculate_llm_price

from openai import OpenAI

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
documents[6]

{'content': '# The LLM\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=KHePGkeFn54&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe last component of our RAG pipeline is the LLM. It takes\nthe prompt we built and generates an answer.\n\n## Sending the prompt to the LLM\n\nWe have the prompt from the previous section.\n\nWe send it to the\nLLM:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=prompt\n)\n```\n\nWe use OpenAI\'s Responses API (`openai_client.responses.create`). OpenAI\nhas two APIs: chat completions and responses. Chat completions is the\nolder one, and it\'s now considered legacy. When the first edition of\nthis course started, the Responses API didn\'t exist, so we used chat\ncompletions. Now we prefer responses because it\'s more convenient.\n\nThere\'s a catch worth knowing. Many other providers like Groq and\nGemini give you an OpenAI-compatible client. But they expose chat\ncompletions, not responses. So if 

In [5]:
len(files)

72

In [6]:
minsearch_index = build_index(documents=documents, text_fields=["content"], keyword_fields=["filename"])
minsearch_index

In [7]:
query="How does the agentic loop keep calling the model until it stops?"

In [8]:
minsearch_index.search(query, num_results=1)[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [15]:
openai_client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    # base_url="https://openrouter.ai/api/v1",
    # api_key=os.getenv("OPENROUTER_API_KEY"),
    api_key="1234",
)

In [10]:
assistant = RAGHW(index=minsearch_index, llm_client=openai_client)

In [16]:
response = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [17]:
response.output_text

'The agentic loop keeps calling the model by using a `while` loop that continues until the model returns a response without any function calls. \n\nSpecifically, the loop works as follows:\n*   **Function Call Detection:** For every response from the model, the code checks if it contains any `function_call` entries.\n*   **Execution and Feedback:** If function calls are present, the code runs the corresponding tools, appends the results to the conversation history (memory), and then loops back to call the model again so it can see the results.\n*   **Exit Condition:** The loop only stops (breaks) when the model returns a response that contains no more function calls, indicating it has provided a final answer.'

In [18]:
dict(response)

{'id': 'resp_4ff825c3d40c09c7beda6a5298bff2bed81549a80634093a',
 'created_at': 1782146315.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'google/gemma-4-12b-qat',
 'object': 'response',
 'output': [ResponseReasoningItem(id='rs_2ffaz04kdw2pytzmowqvji', summary=[], type='reasoning', content=[Content(text='\nHow does the agentic loop keep calling the model until it stops?\nUse the provided context to find the answer. If not found, say "I don\'t know."\n\n    *   The section titled "The full agent loop" and "The core agent loop" seems most relevant.\n    *   Looking for keywords: "loop", "calling the model", "stop", "condition".\n\n    *   *Code segment:*\n        ```python\n        while True:\n            ...\n            response = openai_client.responses.create(...)\n            messages.extend(response.output)\n            for item in response.output:\n                if item.type == "function_call":\n                    ...\n        

In [20]:
llm_usage = response.usage
llm_usage

ResponseUsage(input_tokens=7850, input_tokens_details=InputTokensDetails(cached_tokens=7845), output_tokens=509, output_tokens_details=OutputTokensDetails(reasoning_tokens=359), total_tokens=8359)

In [21]:
calculate_llm_price(llm_usage.input_tokens, llm_usage.output_tokens, response.model)

{'input_cost': 0.0011775,
 'output_cost': 0.0003054,
 'total_cost': 0.0014828999999999999,
 'input_tokens': 7850,
 'output_tokens': 509,
 'model_name': 'google/gemma-4-12b-qat'}

In [23]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [24]:
len(chunks)

295

In [27]:
chunks[10]

{'start': 4000,
 'content': "hat the LLM generates.\nThat search step is what gives the LLM the context it needs to answer\ncorrectly.\n\nWhat we just did was naive. I knew in advance which FAQ entry held the\nanswer and pasted it in by hand. What we want instead is to perform\nsearch automatically. We take the student's question, find the most\nrelevant documents, and send those to the LLM.\n\nIn code, it looks like this:\n\n```python\ndef rag(question):\n    search_results = search(question)\n    user_prompt = build_prompt(question, search_results)\n    return llm(user_prompt)\n```\n\nThat's the entire architecture. It comes down to three components.\n\nThe pieces are search, the prompt, and the LLM:\n\n- search\n- prompt\n- LLM\n\n\n```mermaid\nflowchart TD\n    U([User])\n\n    APP[Application]\n\n    DB[(DB)]\n    DOCS[[D1 ... D5]]\n\n    PROMPT[Build Prompt<br/>Question + Context]\n\n    LLM[LLM]\n\n    ANSWER([Answer])\n\n    U -->|Question| APP\n\n    APP -->|Query| DB\n    DB 

In [28]:
minsearch_index_v2 = build_index(documents=chunks, text_fields=["content"], keyword_fields=["filename"])
minsearch_index_v2

In [29]:
assistant_v2 = RAGHW(index=minsearch_index_v2, llm_client=openai_client)

In [30]:
response_v2 = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [31]:
llm_usage = response_v2.usage
calculate_llm_price(llm_usage.input_tokens, llm_usage.output_tokens, response_v2.model)

{'input_cost': 0.0011775,
 'output_cost': 0.000285,
 'total_cost': 0.0014624999999999998,
 'input_tokens': 7850,
 'output_tokens': 475,
 'model_name': 'google/gemma-4-12b-qat'}